# Tutorial 04: The CRAG State Machine (Wiring the Loop)

In our previous tutorials, we built isolated components: a Semantic Router to protect the database, a Vector Store to retrieve documents, and a Grader to filter out hallucinations.

However, standard LangChain pipelines (LCEL) are linear (`A -> B -> C`). If the Grader rejects all documents, a linear chain fails. To build a Corrective RAG (CRAG) system, we need a system that can make decisions, loop back, and retry. We need a **State Machine**.

In this notebook, we will use `langgraph` to wire our isolated components into an autonomous, self-correcting loop.

### Prerequisites
* Completion of Tutorials 01, 02, and 03.
* Basic understanding of graph theory (Nodes and Edges).
* The `langgraph` library installed (`pip install langgraph`).


### Step 1: Defining the Agent State

In a LangGraph state machine, every node reads from and writes to a shared `State` object. This replaces the fragile process of passing variables explicitly between functions.


In [ ]:
from typing import List, TypedDict

# 1. Define the State schema
class GraphState(TypedDict):
    """
    Represents the state of our CRAG graph.
    
    Attributes:
        question: The user's current query (might be rewritten).
        documents: List of retrieved and graded documents.
        generation: The final LLM response.
    """
    question: str
    documents: List[str]
    generation: str

print("✅ Graph State defined.")


### Step 2: Defining the Nodes (The Actions)

Nodes are standard Python functions that take the `State`, perform an action (like retrieving or grading), and return an updated `State`.

*Note: For this tutorial, we will use simulated functions to represent the logic we built in Tutorials 01-03.*


In [ ]:
def retrieve_node(state: GraphState):
    """Simulates querying the vector database."""
    print("---NODE: RETRIEVE---")
    question = state["question"]
    # Simulated retrieval: We intentionally return one good and one bad document
    docs = [
        "The system uses asymmetric hardware fencing to save VRAM.",
        "The espresso machine needs to be cleaned daily."
    ]
    return {"documents": docs, "question": question}

def grade_documents_node(state: GraphState):
    """Simulates the Grader from Tutorial 03."""
    print("---NODE: GRADE DOCUMENTS---")
    question = state["question"]
    documents = state["documents"]
    
    filtered_docs = []
    for doc in documents:
        # Simulated LLM grading logic: keep only docs mentioning 'VRAM' or 'hardware'
        if "hardware" in doc or "VRAM" in doc:
            print(f"  [Grader] KEEPING relevant doc: {doc}")
            filtered_docs.append(doc)
        else:
            print(f"  [Grader] DROPPING irrelevant doc: {doc}")
            
    return {"documents": filtered_docs, "question": question}

def rewrite_query_node(state: GraphState):
    """Fallback node: Rewrites the query if retrieval failed."""
    print("---NODE: REWRITE QUERY---")
    question = state["question"]
    # Simulated LLM rewrite logic
    better_question = f"hardware specifications and VRAM constraints for {question}"
    print(f"  [Rewrite] Transformed '{question}' -> '{better_question}'")
    return {"question": better_question, "documents": state["documents"]}

def generate_node(state: GraphState):
    """Simulates final LLM generation."""
    print("---NODE: GENERATE---")
    return {"generation": "Simulated final answer based strictly on filtered context.", "question": state["question"], "documents": state["documents"]}


### Step 3: Defining the Edges (The Logic)

Edges control the flow. "Conditional Edges" are where the real power of CRAG lies. They inspect the state and decide which node to trigger next.


In [ ]:
def decide_to_generate(state: GraphState):
    """
    Determines if we have enough good context to generate an answer, 
    or if we need to rewrite the query and try again.
    """
    print("---EDGE: EVALUATE CONTEXT---")
    filtered_documents = state["documents"]
    
    if not filtered_documents:
        # All documents were dropped by the grader. We must rewrite and retry.
        print("  [Decision] No relevant context. Routing to REWRITE.")
        return "rewrite"
    else:
        # We have good context. Proceed to generation.
        print("  [Decision] Context is relevant. Routing to GENERATE.")
        return "generate"


### Step 4: Compiling the Graph

Now, we wire the nodes and edges together into a compiled application.


In [ ]:
from langgraph.graph import StateGraph, END

# 1. Initialize the graph
workflow = StateGraph(GraphState)

# 2. Add the nodes
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("grade_documents", grade_documents_node)
workflow.add_node("rewrite_query", rewrite_query_node)
workflow.add_node("generate", generate_node)

# 3. Define the linear flow
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_documents")

# 4. Define the conditional routing (The CRAG Loop)
workflow.add_conditional_edges(
    "grade_documents", # Starting node
    decide_to_generate, # The conditional function
    {
        "generate": "generate", # If function returns 'generate', go to generate node
        "rewrite": "rewrite_query" # If function returns 'rewrite', go to rewrite node
    }
)

# 5. Close the loop
workflow.add_edge("rewrite_query", "retrieve") # After rewriting, search again!
workflow.add_edge("generate", END)

# 6. Compile
app = workflow.compile()
print("✅ CRAG State Machine compiled successfully.")


### Step 5: Executing the Self-Correcting Loop

Watch the terminal output closely. You will see the system retrieve documents, filter out the bad one (the espresso machine), and proceed to generation.

If we altered the `retrieve_node` to return *only* the bad document, you would see the graph autonomously route to `rewrite_query`, transform the question, and loop back to `retrieve` without writing a single `while True` loop.


In [ ]:
# Run the application
inputs = {"question": "How do we prevent VRAM overflow?"}
print(f"Initial User Query: {inputs['question']}\n")

final_val = None
for output in app.stream(inputs):
    for key, value in output.items():
        final_val = value

if final_val:
    print(f"\nFinal System Generation: {final_val.get('generation')}")


### The Architectural Win

You have just built a non-deterministic, self-healing architecture. If an LLM or vector search fails to provide quality data, the application does not crash or hallucinate—it identifies the failure, executes a fallback strategy, and tries a different approach. This cyclic resiliency is what separates enterprise AI agents from basic prototypes.

*Note: You can inspect the production version of this state machine with dependency injection inside [`scripts/crag_graph.py`](../scripts/crag_graph.py).*

---

### 🏋️ Challenge

Our current graph has a dangerous flaw: if the `rewrite_query` node consistently fails to produce a query that yields good documents, the system will loop between `retrieve` -> `grade` -> `rewrite` infinitely.

**Your Task:**
Update the `GraphState` `TypedDict` to include a `retry_count: int`. Update the nodes to increment this counter. Finally, update the `decide_to_generate` conditional edge so that if `retry_count > 3`, it routes directly to a new node called `generate_fallback` (which tells the user "I'm sorry, I don't have that information") instead of looping forever.

In **Tutorial 05: Hardware-Aware Fencing**, we will step away from LangGraph and look at the physical deployment layer, exploring how to execute this pipeline without overflowing your GPU's VRAM.
